# Merkle DE Case Study – three-layer data lake & `top_item` datamart

This notebook runs every step of the hands-on assignment on **Databricks Free Edition** (serverless compute, Unity
Catalog), including all DDLs. Run it with *Run all*. Each run rebuilds all tables from scratch.

| Layer | Schema | Tables | Content |
|---|---|---|---|
| 1 – raw | `layer1_raw` | `item`, `event` | Source columns as-is, all `STRING`, no transformations |
| 2 – curated | `layer2_curated` | `item`, `event` | snake_case names, JSON payload flattened, proper types, `event` partitioned by `event_year` |
| 3 – mart | `layer3_mart` | `top_item` | Views, rank and most used platform(s) per item and year |

**Sources**

* `item.csv` is the product catalogue (dimension): 2,198 items.
* `event.csv` is a tracking log (fact table). The file has **one row per event parameter**: an event (`event_id`) has
  1–2 rows, and the JSON column `event.payload` holds the event name, platform and one parameter name/value.

The profiling behind these findings is in `data_exploration.ipynb`.

## 0. Configuration

All names used in the notebook are defined here: the catalog, the source URL, the landing path and the five tables.

In [1]:
from pyspark.sql import functions as F

CATALOG = "workspace"   # default Unity Catalog catalog in Databricks Free Edition
SOURCE_URL = "https://merkle-de-interview-case-study.s3.eu-central-1.amazonaws.com/de"
LANDING_PATH = f"/Volumes/{CATALOG}/layer1_raw/landing"

L1_ITEM  = f"{CATALOG}.layer1_raw.item"
L1_EVENT = f"{CATALOG}.layer1_raw.event"
L2_ITEM  = f"{CATALOG}.layer2_curated.item"
L2_EVENT = f"{CATALOG}.layer2_curated.event"
TOP_ITEM = f"{CATALOG}.layer3_mart.top_item"

## 1. Schemas and landing volume

There is one schema per layer. The `landing` volume is needed because serverless `spark.read` cannot read `https://`
URLs. The files are first copied into the volume unchanged.

In [2]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.layer1_raw COMMENT 'Layer 1: raw source data, all STRING, no transformations'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.layer2_curated COMMENT 'Layer 2: curated data - renamed, flattened, typed, partitioned'")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.layer3_mart COMMENT 'Layer 3: datamarts'")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.layer1_raw.landing COMMENT 'Source files downloaded from S3'")

## 2. Load the source files from S3

Both files are downloaded from the public bucket into the landing volume unchanged.

In [3]:
import urllib.request

for file_name in ["item.csv", "event.csv"]:
    urllib.request.urlretrieve(f"{SOURCE_URL}/{file_name}", f"{LANDING_PATH}/{file_name}")

for f in dbutils.fs.ls(LANDING_PATH):
    print(f"{f.name}: {f.size:,} bytes")

event.csv: 155,970,517 bytes
item.csv: 184,207 bytes


## 3. Layer 1 – raw tables

Column names match the file headers exactly, including the dotted `event.payload`, and every column is `STRING`.

Two reader options matter:

* **No schema inference.** Spark then reads every column as `STRING`. Empty fields (e.g. a missing `adjective`)
  are stored as `NULL`.
* **`escape = '"'`.** The JSON in `event.payload` escapes quotes by doubling them (`""`). Spark's default escape
  character is `\`, which would split the payload across columns.

In [4]:
spark.sql(f"""
CREATE OR REPLACE TABLE {L1_ITEM} (
    adjective   STRING,
    category    STRING,
    created_at  STRING,
    id          STRING,
    modifier    STRING,
    name        STRING,
    price       STRING
)
COMMENT 'Layer 1: raw copy of item.csv'
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {L1_EVENT} (
    event_id         STRING,
    event_time       STRING,
    user_id          STRING,
    `event.payload`  STRING
)
COMMENT 'Layer 1: raw copy of event.csv (one row per event parameter)'
""")

In [5]:
# Read both files without schema inference (every column STRING) and load them into layer 1
raw_item = spark.read.csv(f"{LANDING_PATH}/item.csv", header=True, escape='"')
raw_event = spark.read.csv(f"{LANDING_PATH}/event.csv", header=True, escape='"')

# insertInto matches columns by position, so the columns are selected by name in DDL order:
# a renamed or missing source column fails here instead of shifting values between columns
raw_item.select(
    "adjective",
    "category",
    "created_at",
    "id",
    "modifier",
    "name",
    "price",
).write.insertInto(L1_ITEM, overwrite=True)

raw_event.select(
    "event_id",
    "event_time",
    "user_id",
    "`event.payload`",
).write.insertInto(L1_EVENT, overwrite=True)

display(spark.table(L1_EVENT).limit(5))

event_id,event_time,user_id,event.payload
ec5014f8a3aa402fbb207b22fac9121a,2014-06-16 15:23:05,110892.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
ebf374a7668945db83cccc9e6adc86db,2014-05-17 15:23:05,272356.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
bded8cc52707405c9dabe6452d20f8d6,2014-06-23 15:23:05,213337.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
5bb6b791701b4ca8a435ed247fd14a00,2014-05-30 15:23:05,42912.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"
5c59cb9206f043b7bb9da75186f88b7e,2014-05-27 15:23:05,139790.0,"{""event_name"":""test_assignment"",""platform"":""server"",""parameter_name"":""test_id"",""parameter_value"":""6""}"


## 4. Layer 2 – curated tables

**Naming convention:** all names are `snake_case`, and identifiers are called `<entity>_id` (`id` → `item_id`).
Descriptive attributes keep their source names.

**Data types**

* **IDs** become `INT`.
  * `item.id` and `user_id` are always written as `"2512.0"`. With ANSI mode on (the serverless default), a direct
    `CAST('2512.0' AS INT)` fails.
  * The literal `.0` suffix is therefore removed before casting. Going through `DOUBLE` would silently truncate a
    value like `2512.5`; this way such a value still fails the cast.
  * IDs inside the payload are plain digits and are cast directly.
* **`price`** becomes `DECIMAL(10,2)`. The source carries floating-point noise (e.g. `213.60000000000002`), and the
  interviewer confirmed that their systems operate on two decimal places. Values are rounded half-up.
* **`created_at` and `event_time`** become `TIMESTAMP_NTZ`. The source has no time zone, so the values are stored
  exactly as provided rather than read in a session time zone.
* **`test_assignment`** becomes `INT`, not `BOOLEAN`, because `0`/`1` are experiment variants.

**Partitioning:** the fact table `event` is partitioned by `event_year`.
* The datamart aggregates by year.
* Daily partitions would mean about 1,900 tiny partitions for this small dataset.

In [6]:
spark.sql(f"""
CREATE OR REPLACE TABLE {L2_ITEM} (
    item_id     INT            NOT NULL COMMENT 'Item identifier (source: id)',
    item_name   STRING                  COMMENT 'adjective + category + modifier; not unique (source: name)',
    adjective   STRING                  COMMENT 'e.g. fuzzy; may be NULL',
    category    STRING                  COMMENT 'e.g. gadget',
    modifier    STRING                  COMMENT 'e.g. charger; may be NULL',
    price       DECIMAL(10,2)           COMMENT 'Item price',
    created_at  TIMESTAMP_NTZ           COMMENT 'Item creation time as provided (no time zone)'
)
COMMENT 'Layer 2: item dimension, one row per item'
""")

spark.sql(f"""
CREATE OR REPLACE TABLE {L2_EVENT} (
    event_id         STRING         NOT NULL COMMENT 'Event identifier',
    event_time       TIMESTAMP_NTZ  NOT NULL COMMENT 'Event time as provided (no time zone)',
    event_year       INT            NOT NULL COMMENT 'Year of event_time; partition column',
    user_id          INT                     COMMENT 'User who triggered the event',
    event_name       STRING                  COMMENT 'view_item | view_user_profile | test_assignment',
    platform         STRING                  COMMENT 'web | android | iOS | mobile web | server',
    item_id          INT                     COMMENT 'Viewed item (view_item only)',
    referrer         STRING                  COMMENT 'Where the item view came from (view_item only)',
    test_id          INT                     COMMENT 'A/B test id (test_assignment only)',
    test_assignment  INT                     COMMENT 'A/B test variant 0/1 (test_assignment only)',
    viewed_user_id   INT                     COMMENT 'Viewed user profile (view_user_profile only)'
)
PARTITIONED BY (event_year)
COMMENT 'Layer 2: event fact, one row per event'
""")

### 4.1 `item`

`item` has no nested attributes, so this step only renames and casts.

In [7]:
item = spark.table(L1_ITEM).select(
    F.regexp_replace("id", r"\.0$", "").cast("int").alias("item_id"),   # "2512.0" -> 2512
    F.col("name").alias("item_name"),
    "adjective",
    "category",
    "modifier",
    F.col("price").cast("decimal(10,2)").alias("price"),         # rounds to 2 decimals
    F.col("created_at").cast("timestamp_ntz").alias("created_at"),
)

# insertInto matches columns by position; the select above follows the DDL column order
item.write.insertInto(L2_ITEM, overwrite=True)

### 4.2 `event`: flatten the payload and pivot

`event.payload` is a JSON object with four keys: `event_name`, `platform`, `parameter_name` and `parameter_value`.
Each event type has a fixed set of parameters:

| event_name | parameters (one source row each) |
|---|---|
| `view_item` | `item_id`, `referrer` |
| `test_assignment` | `test_id`, `test_assignment` |
| `view_user_profile` | `viewed_user_id` |

The transformation has three steps:

1. Parse the JSON into columns.
2. Pivot the parameter rows into **one row per event**, so each parameter gets its own column (`NULL` where it doesn't apply).
3. Cast the columns to their types.

Within an event, the event attributes (time, user, name, platform) are identical across rows, and each parameter
appears only once. `first()` therefore loses nothing, which the validation in section 6 confirms.

In [8]:
payload_schema = "event_name STRING, platform STRING, parameter_name STRING, parameter_value STRING"
parameters = ["item_id", "referrer", "test_id", "test_assignment", "viewed_user_id"]

event = (
    spark.table(L1_EVENT)
    .withColumn("payload", F.from_json(F.col("`event.payload`"), payload_schema))
    .select("event_id", "event_time", "user_id", "payload.*")
    .groupBy("event_id", "event_time", "user_id", "event_name", "platform")
    .pivot("parameter_name", parameters)
    .agg(F.first("parameter_value"))
    .select(
        "event_id",
        F.col("event_time").cast("timestamp_ntz").alias("event_time"),
        F.year(F.col("event_time").cast("timestamp_ntz")).alias("event_year"),
        F.regexp_replace("user_id", r"\.0$", "").cast("int").alias("user_id"),   # "178481.0" -> 178481
        "event_name",
        "platform",
        F.col("item_id").cast("int").alias("item_id"),
        "referrer",
        F.col("test_id").cast("int").alias("test_id"),
        F.col("test_assignment").cast("int").alias("test_assignment"),
        F.col("viewed_user_id").cast("int").alias("viewed_user_id"),
    )
)

# insertInto matches columns by position; the select above follows the DDL column order
event.write.insertInto(L2_EVENT, overwrite=True)

display(spark.sql(f"SELECT event_name, platform, COUNT(*) AS events FROM {L2_EVENT} GROUP BY ALL ORDER BY ALL"))

event_name,platform,events
test_assignment,server,162490
view_item,android,43499
view_item,iOS,42982
view_item,mobile web,43364
view_item,web,132941
view_user_profile,android,503
view_user_profile,iOS,550
view_user_profile,mobile web,496
view_user_profile,web,1539


## 5. Layer 3 – `top_item` datamart

### Documentation

**Purpose:** shows how popular each item is in each year: how often it was viewed, where it ranks against the other
items, and on which platform it was viewed most.

**Grain:** one row per `item_id` and `view_year`, **only for combinations with at least one view**.

> **Note for the visualisation team:** item/year combinations without views are not listed, as agreed with the
> interviewer. A missing combination means zero views (for example, 95 items have no views in 2013).

| Attribute | Type | Description |
|---|---|---|
| `item_id` | `INT` | Item identifier; join to `layer2_curated.item` for name, category and price |
| `view_year` | `INT` | Calendar year of the views, from `event_time` as provided (no time zone) |
| `total_views` | `BIGINT` | Number of views of the item in that year |
| `view_rank` | `INT` | Rank within the year by `total_views`, where 1 = most viewed. Items with equal views share a rank, as requested by the interviewer. `RANK()` skips the following ranks (1, 1, 3): the request didn't specify this, and we chose it so that a rank equals 1 + the number of items with more views |
| `most_used_platforms` | `ARRAY<STRING>` | Platform(s) with the most views of the item in that year. If platforms tie, **all** of them are listed, as requested by the interviewer, sorted alphabetically so the output is deterministic (≈8% of rows) |

**Definitions**

* **Item view:** one event with `event_name = 'view_item'`, and the item is its `item_id` parameter. Layer 2 holds
  one row per event, so `COUNT(*)` counts views. On the raw layer 1 rows the same count would double, because each
  view has both an `item_id` row and a `referrer` row.
* **Platform:** the `platform` of the view event (`web`, `android`, `iOS`, `mobile web`).

**Notes for consumers**

* **2013 and 2018 are partial years.** Events run from 2013-03-09 to 2018-06-01, so those two years can't be
  compared with full years.
* **3,148 views come before the item's `created_at`.** They are kept and counted, because the data can't tell
  whether the event time or the creation time is wrong.

### DDL

In [9]:
spark.sql(f"""
CREATE OR REPLACE TABLE {TOP_ITEM} (
    item_id              INT            NOT NULL COMMENT 'Item identifier, joins to layer2_curated.item',
    view_year            INT            NOT NULL COMMENT 'Year of the views (from event_time)',
    total_views          BIGINT         NOT NULL COMMENT 'Number of item views in the year',
    view_rank            INT            NOT NULL COMMENT 'RANK() by total_views within the year; ties share a rank',
    most_used_platforms  ARRAY<STRING>  NOT NULL COMMENT 'Platform(s) with most views in the year; all ties, sorted'
)
COMMENT 'Layer 3: yearly item views, rank and most used platform(s); only item/year combinations with views'
""")

### Load

The query has three parts:

1. `item_year`: counts the views per item and year.
2. `platform_rank`: counts the views per item, year and platform, then ranks the platforms. Rank 1 is the most used
   platform, and tied platforms all get rank 1.
3. Final select: ranks the items within each year and attaches the rank-1 platforms as a sorted array.

In [10]:
top_item = spark.sql(f"""
WITH views AS (
    SELECT item_id, event_year AS view_year, platform
    FROM {L2_EVENT}
    WHERE event_name = 'view_item'
),
item_year AS (
    SELECT item_id, view_year, COUNT(*) AS total_views
    FROM views
    GROUP BY item_id, view_year
),
platform_rank AS (
    SELECT item_id, view_year, platform,
           RANK() OVER (PARTITION BY item_id, view_year ORDER BY COUNT(*) DESC) AS platform_rank
    FROM views
    GROUP BY item_id, view_year, platform
),
top_platforms AS (
    SELECT item_id, view_year, array_sort(collect_list(platform)) AS most_used_platforms
    FROM platform_rank
    WHERE platform_rank = 1
    GROUP BY item_id, view_year
)
SELECT item_id,
       view_year,
       total_views,
       RANK() OVER (PARTITION BY view_year ORDER BY total_views DESC) AS view_rank,
       most_used_platforms
FROM item_year
JOIN top_platforms USING (item_id, view_year)
""")

# insertInto matches columns by position; the select above follows the DDL column order
top_item.write.insertInto(TOP_ITEM, overwrite=True)

In [11]:
# Top 3 items per year, with item names added for readability
display(spark.sql(f"""
    SELECT t.view_year, t.view_rank, t.item_id, i.item_name, t.total_views, t.most_used_platforms
    FROM {TOP_ITEM} t JOIN {L2_ITEM} i USING (item_id)
    WHERE t.view_rank <= 3
    ORDER BY t.view_year, t.view_rank, t.item_id
"""))

view_year,view_rank,item_id,item_name,total_views,most_used_platforms
2013,1,2086,organic device,12,"[""web""]"
2013,1,3517,prize-winning widget wrapper,12,"[""web""]"
2013,3,3063,fuzzy gadget wrapper,11,"[""web""]"
2014,1,244,miniature contraption,29,"[""web""]"
2014,1,1353,mechanism,29,"[""web""]"
2014,3,27,aerodynamic mechanism,28,"[""web""]"
2015,1,55,prize-winning instrument charger,39,"[""web""]"
2015,1,511,extra-strength mechanism storage_unit,39,"[""web""]"
2015,1,1327,extra-strength apparatus charger,39,"[""web""]"
2015,1,2094,analog tool,39,"[""web""]"


## 6. Validation

These checks cover what would make the datamart wrong:

* **No events lost or duplicated.** Layer 2 must have exactly one row per source `event_id`.
* **The pivot kept every parameter.** The parameter values in layer 2 must match the layer 1 row count one for one.
  This also catches an unknown parameter name or a repeated parameter.
* **Views reconcile.** Every view must point to an existing item, and `top_item` must add up to all view events,
  with one row per item and year.

In [12]:
c = spark.sql(f"""
SELECT
    (SELECT COUNT(DISTINCT event_id) FROM {L1_EVENT})                    AS l1_events,
    (SELECT COUNT(*) FROM {L2_EVENT})                                    AS l2_events,
    (SELECT COUNT(DISTINCT event_id) FROM {L2_EVENT})                    AS l2_distinct_events,
    (SELECT COUNT(*) FROM {L1_EVENT})                                    AS l1_parameter_rows,
    -- Each raw parameter row must become exactly one populated parameter column after the pivot,
    -- so the non-NULL parameter values in layer 2 must equal the layer 1 row count
    -- (source parameter values are never empty).
    (SELECT COUNT(item_id) + COUNT(referrer) + COUNT(test_id)
            + COUNT(test_assignment) + COUNT(viewed_user_id)
     FROM {L2_EVENT})                                                    AS l2_parameter_values,
    (SELECT COUNT(*) FROM {L2_EVENT} e LEFT ANTI JOIN {L2_ITEM} i ON e.item_id = i.item_id
     WHERE e.event_name = 'view_item')                                   AS views_without_item,
    (SELECT COUNT(*) FROM {L2_EVENT} WHERE event_name = 'view_item')     AS l2_views,
    (SELECT SUM(total_views) FROM {TOP_ITEM})                            AS top_item_views,
    (SELECT COUNT(*) FROM {TOP_ITEM})                                    AS top_item_rows,
    (SELECT COUNT(DISTINCT item_id, view_year) FROM {TOP_ITEM})          AS top_item_item_years
""").first()
print(c.asDict())

assert c.l2_events == c.l2_distinct_events == c.l1_events, "events lost or duplicated"
assert c.l2_parameter_values == c.l1_parameter_rows,       "parameters lost in the pivot"
assert c.views_without_item == 0,                          "views of unknown items"
assert c.top_item_views == c.l2_views,                     "top_item does not add up to all views"
assert c.top_item_rows == c.top_item_item_years,           "top_item has duplicate item/year rows"
print("All checks passed")

{'l1_events': 428364, 'l2_events': 428364, 'l2_distinct_events': 428364, 'l1_parameter_rows': 853640, 'l2_parameter_values': 853640, 'views_without_item': 0, 'l2_views': 262786, 'top_item_views': 262786, 'top_item_rows': 13093, 'top_item_item_years': 13093}
All checks passed


### Data-quality observations

These anomalies are reported here and not removed (see the datamart notes above).

In [13]:
display(spark.sql(f"""
SELECT e.event_year,
       MIN(e.event_time)                        AS first_view,
       MAX(e.event_time)                        AS last_view,
       COUNT(*)                                 AS views,
       COUNT_IF(e.event_time < i.created_at)    AS views_before_item_created
FROM {L2_EVENT} e JOIN {L2_ITEM} i USING (item_id)
WHERE e.event_name = 'view_item'
GROUP BY e.event_year
ORDER BY e.event_year
"""))

event_year,first_view,last_view,views,views_before_item_created
2013,2013-03-09T09:28:22.000,2013-12-31T22:45:31.000,7448,2888
2014,2014-01-01T00:55:05.000,2014-12-31T23:44:20.000,32082,260
2015,2015-01-01T00:01:44.000,2015-12-31T23:58:08.000,49462,0
2016,2016-01-01T00:02:08.000,2016-12-31T23:59:30.000,63642,0
2017,2017-01-01T00:08:56.000,2017-12-31T23:48:39.000,76314,0
2018,2018-01-01T00:04:14.000,2018-06-01T23:57:54.000,33838,0
